# Euro area GDP nowcast, step by step

This notebook runs the pipeline **step by step** by calling the functions of the `eanowcast` package.
It does not contain the model: it **uses** it. The code lives in `src/eanowcast/*.py`; the notebook is for
exploring, understanding and checking.

| Step | Module | Function called here |
| --- | --- | --- |
| Configuration (series, blocks) | `config.py` + `series.toml` | `load_config` |
| Eurostat download | `eurostat.py`, `data.py` | `download` |
| Vintage database | `store.py` | `Store` |
| Data available at a given date | `data.py` | `Dataset.as_of` |
| Missing months (ragged edge) | `fill.py` | `fill_monthly` |
| Monthly → quarterly, weights | `transforms.py` | `to_quarterly`, `nominal_weights`, `historical_contributions` |
| Bridge per component + aggregation | `nowcast.py` | `fit_block`, `run_nowcast` |
| Benchmarks | `benchmarks.py` | `ar_gdp`, `direct_bridge`, `historical_mean` |
| Evaluation, tracking | `backtest.py` | `backtest`, `metrics`, `track` |
| Chart, table | `report.py` | `tracking_chart`, `format_table` |

**Requirements**: in the repo folder, `pip install -e ".[dev,notebook]"` (or open the notebook in VS Code).

**`USE_SYNTHETIC = True`**: runs on synthetic data, offline, to understand the mechanics.
Switch to `False` after running the download cell (or `eanowcast fetch` in a terminal).

In [ ]:
USE_SYNTHETIC = True        # False = real Eurostat data (SQLite database)
AS_OF = "2026-10-15"        # date at which we place ourselves
TARGET = "2026Q4"           # quarter to nowcast
FILL_METHOD = "ar"          # "ar" (one AR per series) or "dfm" (dynamic factor)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from eanowcast import benchmarks as bm
from eanowcast.backtest import backtest, error_by_component, metrics, track
from eanowcast.config import load_config
from eanowcast.data import download, load_from_store
from eanowcast.fill import fill_monthly
from eanowcast.nowcast import fit_block, run_nowcast
from eanowcast.report import format_table, tracking_chart
from eanowcast.store import Store
from eanowcast.synthetic import make_dataset
from eanowcast.transforms import historical_contributions, nominal_weights, to_quarterly

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "eanowcast.sqlite"
OUT = ROOT / "outputs" / "notebook"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140)

## 1. Configuration

Everything "economic" is declared in `series.toml`: which indicators, which Eurostat dataset,
which transformation, which publication lag, and which regressors for each component.
Adding an indicator = adding a section to that file, without touching the code.

In [ ]:
cfg = load_config()

indicators = pd.DataFrame([{"name": i.name, "label": i.label, "dataset": i.dataset,
                            "transform": i.transform, "lag (days)": i.release_lag_days}
                           for i in cfg.indicators.values()])
blocks = pd.DataFrame([{"block": b.name, "label": b.label, "na_item": b.na_item, "sign": b.sign,
                        "model": b.model, "regressors": ", ".join(b.regressors), "AR(1)": b.ar_lag}
                       for b in cfg.blocks.values()])
display(indicators, blocks)

## 2. Data

- Real data: `download` queries the Eurostat API and writes every new or revised value to the SQLite
  database with the download time. Running `download` again later adds a **vintage**.
- `load_from_store` reads the database back and returns a `Dataset` with three tables:
  `monthly` (raw indicators), `growth` (q/q volume growth, %) and `levels` (current prices, EUR million).

In [ ]:
# Uncomment to download the real data from the notebook (same as `eanowcast fetch`)
# store = Store(DB); download(cfg, store)

if USE_SYNTHETIC:
    ds = make_dataset(cfg)
    print("SYNTHETIC data: the numbers below have no economic meaning.")
else:
    store = Store(DB)
    ds = load_from_store(cfg, store)

display(ds.growth.tail(4).round(2), ds.monthly.tail(4).round(1))

## 3. Placing ourselves at a date: the ragged edge

`ds.as_of(date, cfg)` hides everything that had not been published yet at that date, based on the lags
in `series.toml` (surveys: end of month; retail trade: ~36 days; industrial production: ~45 days;
construction: ~52 days; GDP components: ~65 days). This is what makes pseudo-real-time evaluation possible.

In [ ]:
av = ds.as_of(AS_OF, cfg)

last_month = av.monthly.apply(lambda s: s.last_valid_index())
last_q = av.growth.apply(lambda s: s.last_valid_index())
display(last_month.rename("last published month").to_frame(), last_q.rename("last published quarter").to_frame())

## 4. Extending the missing months (`fill.py`)

Nowcasting a quarter requires its three months for every indicator. Months not yet published are
forecast either by one AR(p) per series (p chosen by AIC) or by a common dynamic factor model
(statsmodels `DynamicFactorMQ`), which for instance uses September's ESI to extend August's industrial
production. This is exactly the role of the factor in GDPNow.

In [ ]:
target = pd.Period(TARGET, freq="Q")
filled = fill_monthly(av.monthly, cfg, until=target.asfreq("M", how="end"), method=FILL_METHOD)

serie = "ip_total"
obs = av.monthly[serie].loc["2025-01":]
fill = filled[serie].loc[obs.last_valid_index():]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(obs.index.to_timestamp(), obs, marker="o", label="observed")
ax.plot(fill.index.to_timestamp(), fill, marker="o", linestyle="--", label=f"extended ({FILL_METHOD})")
ax.set_title(cfg.indicators[serie].label); ax.legend(); plt.show()

## 5. Quarterly conversion and aggregation weights (`transforms.py`)

- Indices (`pct`): average of the three months, then % change on the previous quarter.
- Survey balances (`level`): average of the three months.
- Weights: nominal share of each component in GDP in the previous year.
- Contribution of a component = sign × weight × growth. The **residual** = GDP − sum of contributions:
  it captures the change in inventories and the gap due to chain-linking.

In [ ]:
X = to_quarterly(filled, cfg)
display(X.tail(4).round(2))

names = [b.name for b in cfg.component_blocks]
print("Weights used for", target)
display(nominal_weights(av.levels, target, names).round(3).to_frame("weight"))

contrib = historical_contributions(av.growth, av.levels, cfg)
display(contrib.tail(6).round(2))

## 6. One bridge equation by hand (`nowcast.fit_block`)

Example: household consumption regressed on same-quarter indicators by OLS (statsmodels), from 2000Q1
to the last published quarter, excluding Covid. This is where the model gets judged: signs of the
coefficients, R², stability.

In [ ]:
q_c = av.growth[names].dropna().index.max()       # last quarter published by component
hist = av.growth[names].copy()
hist["residual"] = contrib["residual"].reindex(hist.index)
sample = [q for q in pd.period_range(cfg.start_quarter, q_c, freq="Q") if q not in cfg.exclude_quarters]

res_conso = fit_block(cfg.blocks["consumption"], hist, X, sample)
print(res_conso.summary())

## 7. The full nowcast (`run_nowcast`)

Chains steps 4 to 6 for every block, in the order of `series.toml` (imports use the consumption,
investment and export forecasts), also forecasts the intermediate quarters not yet published (backcast),
then re-aggregates. If the GDP flash is already out for a quarter, the residual is pinned to it.

In [ ]:
res = run_nowcast(av, cfg, TARGET, fill_method=FILL_METHOD)
for q, table in res.by_quarter.items():
    print(f"=== {q} ===")
    print(format_table(table), "\n")

# The estimated models remain available: res.models["exports"].summary(), etc.
print("R² by block:", {k: round(m.rsquared, 2) for k, m in res.models.items()})

## 8. Benchmarks (`benchmarks.py`)

The model is only worth something if it beats these three references, especially the **direct GDP bridge**
(same indicators, no decomposition): it tells whether going through the components adds anything.

In [ ]:
gdp = av.growth["gdp"]
pd.Series({
    "Components (this model)": res.gdp,
    "AR(2) on GDP": bm.ar_gdp(gdp, cfg, target),
    "Direct GDP bridge": bm.direct_bridge(gdp, res.X, cfg, target),
    "Historical mean": bm.historical_mean(gdp, cfg),
}).round(2).to_frame(f"nowcast {TARGET} (%)")

## 9. The GDPNow-style chart (`track` + `tracking_chart`)

The nowcast of the same quarter is rerun every Friday, placing ourselves at each date. Bars show the
contribution of each component, the line the GDP nowcast. This is the chart to show around.

In [ ]:
TRACK_Q = "2026Q2"   # an already published quarter, to compare with the official figure
q = pd.Period(TRACK_Q, freq="Q")
dates = pd.date_range(q.start_time - pd.Timedelta(days=30), q.end_time + pd.Timedelta(days=25), freq="W-FRI")
tr = track(ds, cfg, TRACK_Q, list(dates), fill_method=FILL_METHOD)
actual = ds.growth["gdp"].get(q)
png = tracking_chart(tr, cfg, TRACK_Q, OUT / f"track_{TRACK_Q}.png",
                     actual=None if pd.isna(actual) else float(actual))
display(Image(filename=str(png)))

## 10. Pseudo-real-time evaluation (`backtest.py`)

For each quarter and each horizon (60 and 30 days before quarter end, quarter end, 25 days after),
the available data are rebuilt, everything is re-estimated, and the nowcast is compared with published GDP.
Allow ~1 minute for 10 years with the `ar` method; `dfm` is slower.

In [ ]:
BT_START, BT_END = "2022Q1", "2026Q2"   # widen to 2010Q1 once everything runs
bt = backtest(ds, cfg, BT_START, BT_END, fill_method=FILL_METHOD, verbose=False)
m = metrics(bt)
display(m.round(3))
display(error_by_component(bt, cfg).round(3))

## Outside the notebook

The same steps exist on the command line, for daily use and automation (Docker, Airflow):

| Command line | Equivalent in this notebook |
| --- | --- |
| `eanowcast fetch` | section 2 (`download`) |
| `eanowcast nowcast --save` | section 7, with the result saved to the database |
| `eanowcast track --target 2026Q3` | section 9 |
| `eanowcast backtest --start 2010Q1` | section 10 |
| `eanowcast history --target 2026Q4` | chart of the nowcasts actually saved day after day |

Simple rule: **explore in the notebook, change the model in the `.py` files**, then rerun the notebook.
A notebook that holds the model always ends up drifting away from the production code.